# Oppitunti 11 - Agentista agenttiin (A2A) -protokolla


## Asetus


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv

In [ ]:
import os
import dotenv
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Mikä on A2A-protokolla?

**Agent-to-Agent (A2A) -protokolla** on avoin standardi, joka mahdollistaa tekoälyagenttien välisen viestinnän,
toistensa löytäminen ja yhteistyön – jopa silloin, kun ne on rakennettu eri puitteisiin tai isännöity eri palveluissa.


Keskeiset käsitteet:

- **Löytäminen** – Agentit julkaisevat *Agenttikortin*, joka kuvaa niiden kyvykkyydet, tehden siitä
  helppoa muille agenteille (tai orkestroijille) löytää oikea asiantuntija tehtävään.
- **Viestinvälitys** – Agentit vaihtavat jäsenneltyjä viestejä yhteisen protokollan kautta, joten
  yhden agentin pyyntö voidaan ymmärtää ja toteuttaa toisella huolimatta sen sisäisestä
  toteutuksesta.
- **Tehtävän elinkaari** – A2A määrittelee tiloja kuten *lähetetty*, *työn alla*, *suoritettu* ja
  *epäonnistunut*, antaen orkestroijalle täyden näkyvyyden siihen, miten delegoitu tehtävä etenee.

Tässä oppitunnissa simuloimme A2A-tyyppistä yhteistyötä kytkemällä kolme erikoistunutta matka-agenttia
työnkulkuun, jossa kukin agentti tuo oman asiantuntemuksensa ja välittää tulokset seuraavalle.


## Erikoistuneiden matkatoimistojen luominen


In [ ]:
currency_agent = client.as_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = client.as_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = client.as_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## Moniagenttiyhteistyö työnkulun avulla

Yhdistämme kolme agenttia peräkkäiseen työnkulkuun, joka peilaa A2A-viestinvälitystä:

1. **CurrencyExchangeAgent** vastaanottaa käyttäjän pyynnön ja tuottaa valuuttasuosituksen.
2. **ActivityPlannerAgent** vastaanottaa rikastetun kontekstin ja lisää aktiviteettisuositukset.
3. **TravelManagerAgent** yhdistää molemmat syötteet lopulliseksi matkakoosteeksi.


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## A2A:n ymmärtäminen tuotannossa

Tuotantoympäristössä A2A-protokolla avaa tehokkaita rajapalveluiden välisiä skenaarioita:

| Ominaisuus | Kuvaus |
|---|---|
| **Kehysten välinen yhteentoimivuus** | Yhdellä kehysjärjestelmällä luotu agentti voi delegoida tehtäviä toisella A2A-yhteensopivalla kehyksellä rakennetulle agentille, mahdollistaen todellisen organisaatiorajat ylittävän yhteentoimivuuden. |
| **Palvelurajat** | Agentit voivat sijaita erillisissä mikropalveluissa, pilvialueissa tai jopa eri organisaatioissa, mutta silti tehdä yhteistyötä saumattomasti. |
| **Dynaaminen löydettävyys** | Orkestroija voi ajaa aikana kyselyn Agenttikorttirekisteriin löytääkseen parhaiten sopivan asiantuntijan tiettyyn alitehtävään. |
| **Suoratoisto & push-ilmoitukset** | A2A tukee Server-Sent Events (SSE) -tekniikkaa reaaliaikaisiin etenemisilmoituksiin ja push-ilmoituksia pitkään suoritettaville tehtäville. |

Yllä rakentamamme työnkulku on yksinkertaistettu, prosessissa suoritettava versio tästä mallista. Todellisessa
käyttöönotossa jokainen agentti avaisi HTTP-päätepisteen, julkaisisin Agenttikortin ja kommunikoisi
A2A JSON-RPC -protokollan kautta.


## Yhteenveto

Tässä oppitunnissa opit:

1. **Mikä A2A-protokolla on** — avoin standardi agenttien väliseen löytämiseen, viestintään
   ja tehtävien hallintaan.
2. **Miten luoda erikoistuneita agentteja** — valuutanvaihtoagentti, aktiviteettisuunnittelija-agentti
   ja matkahallinnon orkestroija.
3. **Miten yhdistää agentit työnkulkuun** — käyttäen `WorkflowBuilder`ia mallintamaan peräkkäistä
   viestinvälitystä agenttien välillä.
4. **Miten A2A toimii tuotannossa** — mahdollistaa eri kehysten ja palveluiden välinen yhteistyö
   dynaamisella löytämisellä ja reaaliaikaisilla päivityksillä.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Vastuuvapauslauseke**:
Tämä asiakirja on käännetty käyttämällä tekoälypohjaista käännöspalvelua [Co-op Translator](https://github.com/Azure/co-op-translator). Vaikka pyrimme tarkkuuteen, otathan huomioon, että automaattiset käännökset saattavat sisältää virheitä tai epätarkkuuksia. Alkuperäinen asiakirja sen alkuperäiskielellä on virallinen lähde. Tärkeissä asioissa suositellaan ammattimaista ihmiskäännöstä. Emme ole vastuussa tämän käännöksen käytöstä aiheutuvista väärinymmärryksistä tai tulkinnoista.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
